# 3.2 Flow ablation: the direct XGBoost classifier with and without flow
**Purpose.** A secondary comparison (paper Section 5.1): does recent flow add predictive value to the direct classifier beyond occupancy history, calendar and district? On the test year, it is estimated as ΔAP = AP(with flow) − AP(without flow) per horizon, with a paired bootstrap interval (notebook 4.0). It measures incremental predictive value under this model and feature representation, not causality, and a difference near zero would not show that flow carries no information.

**Method.** Variant A (without flow) has 30 inputs: the four relative-occupancy lags, slot of day as sine and cosine, and the weekday and district dummies. Variant B (with flow) adds the four relative-flow lags (34 inputs) and is the direct XGBoost classifier of notebook 3.1, used as it is. Variant A receives the same 30 candidates, search space and 2023-AP selection rule, and it is free to keep different settings.

**Outputs.** Variant A's settings go to `models/xgboost-ablation.json`, its trials to `models/xgboost-ablation-trials.csv` and its scores to `data/interim/scores-ablation.parquet`. 2024 is not read.

## 1. Setup and rows

In [1]:
import time

import polars as pl

from madrid_congestion.config import (
    HORIZONS,
    INTERIM_DATA_DIR,
    MODELS_DIR,
    PANEL_YEAR,
    PROCESSED_DATA_DIR,
    VALIDATION_YEAR,
)
from madrid_congestion.evaluation import warning_table
from madrid_congestion.features import origin_rows
from madrid_congestion.frozen import check_design, load_frozen, save_frozen
from madrid_congestion.labels import add_labels, references, studied_districts, year_grid
from madrid_congestion.models import (
    SEARCH_SPACE,
    TRIALS,
    at_edge,
    fit_model,
    model_inputs,
    score_model,
    tune_xgboost,
    xgboost_candidates,
)

VARIANT = "direct_no_flow"  # variant A; variant B is the direct model of notebook 3.1
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
series = {year: pl.read_parquet(PROCESSED_DATA_DIR / f"district-series-{year}.parquet") for year in (PANEL_YEAR, VALIDATION_YEAR)}
districts = studied_districts(series[PANEL_YEAR])
check_design(districts)  # the code still builds the frozen design
refs = references(year_grid(series[PANEL_YEAR], districts, PANEL_YEAR))
labels = {year: add_labels(year_grid(df, districts, year), refs) for year, df in series.items()}
rows = {(year, minutes): origin_rows(df, districts, minutes) for year, df in labels.items() for minutes in HORIZONS}

# variant A's inputs are variant B's without the four flow lags (model_inputs in models.py)
without, with_flow = model_inputs(VARIANT, districts), model_inputs("direct", districts)
assert set(with_flow) - set(without) == {f"flow_lag{k}" for k in range(4)} and len(without) == 30
print(len(without), "inputs without flow,", len(with_flow), "with |", {key: df.height for key, df in rows.items()}, "rows")

30 inputs without flow, 34 with | {(2022, 30): 475434, (2022, 60): 453544, (2023, 30): 482391, (2023, 60): 461884} rows


## 2. Tuning variant A on 2023
The same candidates and selection rule as every other XGBoost model (`tune_xgboost`). Each candidate is fitted on 2022 without the flow lags and judged by its 2023 AP.

In [2]:
candidates = xgboost_candidates()
models, trials, selected = {}, {}, {}
for minutes in HORIZONS:
    start = time.time()
    models[minutes], trials[minutes], record = tune_xgboost(
        VARIANT, minutes, rows[PANEL_YEAR, minutes], rows[VALIDATION_YEAR, minutes], districts, candidates
    )
    selected[str(minutes)] = {VARIANT: record}
    print(f"{minutes} min, without flow: {TRIALS} candidates in {time.time() - start:.0f} s")

30 min, without flow: 30 candidates in 274 s
60 min, without flow: 30 candidates in 236 s


**Record of both variants.** The kept settings of variants A and B, whether a kept integer setting lies at an edge of the search space, and a refit of variant A from its saved settings, which must give identical 2023 scores.

In [3]:
variant_b = load_frozen("xgboost-selected.json")
record = []
for minutes in HORIZONS:
    for variant, kept in (("A, without flow", selected[str(minutes)][VARIANT]), ("B, with flow", variant_b[str(minutes)]["direct"])):
        record.append({"minutes": minutes, "variant": variant, **kept, "at_edge": ", ".join(at_edge(kept)) or "none"})
    params = {name: selected[str(minutes)][VARIANT][name] for name in SEARCH_SPACE}
    refit = fit_model("xgboost", VARIANT, minutes, rows[PANEL_YEAR, minutes], districts, params)
    validation = rows[VALIDATION_YEAR, minutes]
    assert (score_model(refit, VARIANT, validation, districts) == score_model(models[minutes], VARIANT, validation, districts)).all()
print("variant A refits to identical 2023 scores at both horizons")
pl.DataFrame(record).with_columns(pl.col("learning_rate", "subsample").round(3), pl.col("ap_2023").round(4))

variant A refits to identical 2023 scores at both horizons


minutes,variant,trial,max_depth,learning_rate,n_estimators,subsample,min_child_weight,ap_2023,at_edge
i64,str,i64,i64,f64,i64,f64,i64,f64,str
30,"""A, without flow""",17,5,0.042,744,0.858,1,0.716,"""min_child_weight"""
30,"""B, with flow""",17,5,0.042,744,0.858,1,0.7202,"""min_child_weight"""
60,"""A, without flow""",13,5,0.08,787,0.659,16,0.7761,"""none"""
60,"""B, with flow""",1,8,0.047,709,0.931,9,0.7822,"""max_depth"""


## 3. The difference on 2023
ΔAP = AP(with flow) − AP(without flow) on the 2023 usable calm origins. Both variants chose their settings on 2023, so the difference is a validation observation without an interval. The 2022 AP is shown for comparison only.

In [4]:
scores_b = pl.read_parquet(INTERIM_DATA_DIR / "scores-xgboost.parquet")
result = []
for minutes in HORIZONS:
    row = {"minutes": minutes}
    for year in (PANEL_YEAR, VALIDATION_YEAR):
        df = rows[year, minutes]
        b = scores_b.filter(pl.col("year") == year, pl.col("minutes") == minutes)
        assert b.select("distrito", "slot_start").equals(df.select("distrito", "slot_start"))  # the same origins, same order
        table = warning_table(df[f"target_{minutes}"], {"without": score_model(models[minutes], VARIANT, df, districts), "with": b["xgboost_direct"]})
        ap = dict(zip(table["score"], table["ap"], strict=True))
        row |= {f"ap_without_{year}": ap["without"], f"ap_with_{year}": ap["with"]}
    row["difference_2023"] = row[f"ap_with_{VALIDATION_YEAR}"] - row[f"ap_without_{VALIDATION_YEAR}"]
    result.append(row)
pl.DataFrame(result).with_columns(pl.exclude("minutes").round(4))

minutes,ap_without_2022,ap_with_2022,ap_without_2023,ap_with_2023,difference_2023
i64,f64,f64,f64,f64,f64
30,0.8151,0.8348,0.716,0.7202,0.0042
60,0.8472,0.9059,0.7761,0.7822,0.0061


## 4. Frozen outputs

In [5]:
save_frozen("xgboost-ablation.json", selected)
pl.concat([table.select(pl.lit(minutes).alias("minutes"), pl.lit(VARIANT).alias("route"), pl.all()) for minutes, table in trials.items()]).write_csv(
    MODELS_DIR / "xgboost-ablation-trials.csv"
)
pl.concat([
    df.select(
        pl.lit(year).alias("year"), pl.lit(minutes).alias("minutes"), "distrito", "slot_start", target=pl.col(f"target_{minutes}"),
        xgboost_direct_no_flow=pl.Series(score_model(models[minutes], VARIANT, df, districts)),
    )
    for (year, minutes), df in rows.items()
]).write_parquet(INTERIM_DATA_DIR / "scores-ablation.parquet")
print(sorted(path.name for path in MODELS_DIR.iterdir()))

['calendar-2022.parquet', 'design.json', 'references-2022.parquet', 'thresholds-2023.json', 'xgboost-ablation-trials.csv', 'xgboost-ablation.json', 'xgboost-selected.json', 'xgboost-trials.csv']


**Result (2023 validation, not test results).** At 30 minutes, variant A keeps trial 17, the same settings as variant B. At 60 minutes, it keeps trial 13, while variant B keeps trial 1. Variant A refits to identical 2023 scores at both horizons. On 2023, the variant with flow has a slightly higher AP, 0.7202 against 0.7160 at 30 minutes (+0.0042) and 0.7822 against 0.7761 at 60 minutes (+0.0061). At 60 minutes, this difference reflects the features and the different settings together. Both values come from the year that tuned the two variants, so they are optimistic and carry no interval.

**Conclusion.** On the validation year, flow added at most a small amount of AP. These are not final-test conclusions. The comparison with intervals is made once, on 2024, in notebook 4.0.